- DSC540-T303 Data Preparation  
- Term Project, Milestone 4  
- Shelia Howard  

# DSC540 Term Project Milestone 4: API Data Cleaning and Formatting

## Introduction

This notebook completes Term Project Milestone 4 by connecting directly to the CDC PLACES public API, pulling county-level chronic disease data in JSON format, and cleaning the API data into a human-readable dataset. The project focuses on healthcare utilization context by preparing chronic disease indicators that can later be compared with utilization or healthcare infrastructure data from prior milestones.

Throughout the notebook, each code block includes comments explaining what the code is doing and why the step matters.

The notebook includes the following sections:

1. Import required Python libraries.
2. Define the API endpoint and query parameters.
3. Pull data directly from the CDC API/JSON source.
4. Perform clearly labeled data transformation and cleansing steps.
5. Print a final human-readable dataset.
6. Discuss ethical implications of the data wrangling process.
7. Conclude with connections to data preparation concepts from the course.

## Important connection issue resolved

The prior version attempted to pull CMS Hospital General Information through a CMS endpoint that repeatedly returned HTTP errors. Because Milestone 4 specifically requires API/JSON cleaning, this revised notebook focuses on the CDC PLACES API source originally identified for the API portion of the project. CMS and hospital infrastructure data can still remain part of the broader term project through prior flat-file or website-based milestones.


In [1]:
# Import the requests library so Python can send an HTTP request to the public CDC API.
import requests

# Import pandas so the JSON API response can be converted into a DataFrame for cleaning.
import pandas as pd

# Import numpy so the notebook can handle missing values and numeric calculations.
import numpy as np

# Import display so selected DataFrames can be shown clearly in the notebook.
from IPython.display import display


### Why this code block matters

This block imports the libraries needed for API access, JSON-to-DataFrame conversion, data cleaning, and readable notebook output. These imports support the core data preparation workflow for this milestone.


In [2]:
# Store the CDC PLACES county-level API endpoint for the 2025 release.
cdc_places_url = "https://data.cdc.gov/resource/swc5-untb.json"

# Create a list of chronic disease and health outcome measures that support the project topic.
target_measures = [
    "DIABETES",
    "OBESITY",
    "BPHIGH",
    "CHD",
    "STROKE"
]

# Create a dictionary of API query parameters for the CDC Socrata API.
cdc_params = {
    # Limit the number of records returned so the notebook stays manageable.
    "$limit": 50000,

    # Keep only county-level records for the selected chronic disease measures.
    "$where": "measureid in('DIABETES','OBESITY','BPHIGH','CHD','STROKE')",

    # Sort the API results so the raw output is easier to review.
    "$order": "statedesc, locationname, measureid"
}

# Print the API endpoint so the source of the data is documented in the notebook output.
print("CDC PLACES API endpoint:", cdc_places_url)

# Print the selected measures so the scope of the API pull is transparent.
print("Selected chronic disease measures:", target_measures)


CDC PLACES API endpoint: https://data.cdc.gov/resource/swc5-untb.json
Selected chronic disease measures: ['DIABETES', 'OBESITY', 'BPHIGH', 'CHD', 'STROKE']


### Why this code block matters

This block defines the API source and the measures used for the project. The notebook uses CDC PLACES because the project proposal identified CDC chronic disease prevalence data as the API source. The selected measures provide county-level health context for diabetes, obesity, blood pressure, heart disease, and stroke.


In [3]:
# Define a reusable function to request JSON data from a public API.
def get_json_from_api(url, params=None, source_name="API source"):
    # Send a GET request to the API using the supplied URL and parameters.
    response = requests.get(url, params=params, timeout=60)

    # Print the final URL so the exact API request is documented for reproducibility.
    print(f"{source_name} request URL:")
    print(response.url)

    # Print the status code so connection problems are visible before cleaning begins.
    print(f"{source_name} status code:", response.status_code)

    # If the request fails, print a preview of the API response to make troubleshooting easier.
    if response.status_code != 200:
        # Print the first part of the response text so the API error message is visible.
        print(response.text[:1000])

        # Stop the notebook with the original HTTP error if the API request was unsuccessful.
        response.raise_for_status()

    # Convert the successful JSON response into a Python object.
    json_data = response.json()

    # Return the JSON object so it can be converted into a pandas DataFrame.
    return json_data


### Why this code block matters

This function makes the API pull more transparent. It prints the request URL and status code before any cleaning happens. That matters because API data preparation should start by confirming that the connection worked and that the notebook is using the intended source.


In [4]:
# Request CDC PLACES data directly from the public API/JSON source.
cdc_json = get_json_from_api(cdc_places_url, params=cdc_params, source_name="CDC PLACES")

# Convert the CDC JSON list into a pandas DataFrame for cleaning and transformation.
cdc_raw = pd.DataFrame(cdc_json)

# Confirm the raw CDC DataFrame shape before any cleaning steps are applied.
print("CDC raw shape before cleaning:", cdc_raw.shape)

# Print the available CDC columns so the original API structure is documented.
print("CDC raw columns:")
print(cdc_raw.columns.tolist())

# Display the first few records so the raw API data can be visually inspected.
display(cdc_raw.head())


CDC PLACES request URL:
https://data.cdc.gov/resource/swc5-untb.json?%24limit=50000&%24where=measureid+in%28%27DIABETES%27%2C%27OBESITY%27%2C%27BPHIGH%27%2C%27CHD%27%2C%27STROKE%27%29&%24order=statedesc%2C+locationname%2C+measureid
CDC PLACES status code: 200
CDC raw shape before cleaning: (29580, 22)
CDC raw columns:
['year', 'stateabbr', 'statedesc', 'locationname', 'datasource', 'category', 'measure', 'data_value_unit', 'data_value_type', 'data_value', 'low_confidence_limit', 'high_confidence_limit', 'totalpopulation', 'totalpop18plus', 'locationid', 'categoryid', 'measureid', 'datavaluetypeid', 'short_question_text', 'geolocation', 'data_value_footnote_symbol', 'data_value_footnote']


,year,stateabbr,statedesc,locationname,datasource,category,measure,data_value_unit,data_value_type,data_value,...,totalpopulation,totalpop18plus,locationid,categoryid,measureid,datavaluetypeid,short_question_text,geolocation,data_value_footnote_symbol,data_value_footnote
0,2023,AL,Alabama,Autauga,BRFSS,Health Outcomes,High blood pressure among adults,%,Age-adjusted prevalence,38.2,...,60342,46253,01001,HLTHOUT,BPHIGH,AgeAdjPrv,High Blood Pressure,"{'type': 'Point', 'coordinates': [-86.64281641...",NaN,NaN
1,2023,AL,Alabama,Autauga,BRFSS,Health Outcomes,High blood pressure among adults,%,Crude prevalence,42.0,...,60342,46253,01001,HLTHOUT,BPHIGH,CrdPrv,High Blood Pressure,"{'type': 'Point', 'coordinates': [-86.64281641...",NaN,NaN
2,2023,AL,Alabama,Autauga,BRFSS,Health Outcomes,Coronary heart disease among adults,%,Age-adjusted prevalence,5.6,...,60342,46253,01001,HLTHOUT,CHD,AgeAdjPrv,Coronary Heart Disease,"{'type': 'Point', 'coordinates': [-86.64281641...",NaN,NaN
3,2023,AL,Alabama,Autauga,BRFSS,Health Outcomes,Coronary heart disease among adults,%,Crude prevalence,6.7,...,60342,46253,01001,HLTHOUT,CHD,CrdPrv,Coronary Heart Disease,"{'type': 'Point', 'coordinates': [-86.64281641...",NaN,NaN
4,2023,AL,Alabama,Autauga,BRFSS,Health Outcomes,Diagnosed diabetes among adults,%,Age-adjusted prevalence,11.4,...,60342,46253,01001,HLTHOUT,DIABETES,AgeAdjPrv,Diabetes,"{'type': 'Point', 'coordinates': [-86.64281641...",NaN,NaN


### Why this code block matters

This block pulls the API data directly from the CDC JSON source and converts it into a DataFrame. The professor's instructions require working directly from the API/JSON source rather than exporting the API data to CSV first. The shape, columns, and preview help verify that the pull was successful before transformations begin.


### Transformation Step #1: Standardize headers

This step standardizes the API field names by trimming spaces, changing names to lowercase, and replacing spaces or hyphens with underscores. Header standardization makes later code easier to write and reduces errors caused by inconsistent column naming.


In [5]:
# Create a copy of the CDC raw DataFrame so the original API pull remains unchanged.
cdc_clean = cdc_raw.copy()

# Standardize CDC column names by removing extra spaces, lowercasing, and replacing spaces or hyphens.
cdc_clean.columns = (
    cdc_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

# Print the cleaned column names to document Transformation Step #1.
print("Transformation Step #1 - CDC columns after header standardization:")
print(cdc_clean.columns.tolist())


Transformation Step #1 - CDC columns after header standardization:
['year', 'stateabbr', 'statedesc', 'locationname', 'datasource', 'category', 'measure', 'data_value_unit', 'data_value_type', 'data_value', 'low_confidence_limit', 'high_confidence_limit', 'totalpopulation', 'totalpop18plus', 'locationid', 'categoryid', 'measureid', 'datavaluetypeid', 'short_question_text', 'geolocation', 'data_value_footnote_symbol', 'data_value_footnote']


### Transformation Step #2: Keep relevant fields and rename columns

This step keeps only the fields needed for the project and renames several columns into more readable names. This improves interpretability because fields like `locationname` and `data_value` are converted into clearer names such as `county_name` and `estimate_percent`.


In [6]:
# Create a list of columns expected from the CDC PLACES county-level API.
expected_columns = [
    "year",
    "stateabbr",
    "statedesc",
    "locationname",
    "locationid",
    "category",
    "measure",
    "measureid",
    "data_value",
    "data_value_unit",
    "data_value_type",
    "low_confidence_limit",
    "high_confidence_limit",
    "totalpopulation"
]

# Keep only the expected columns that are actually present in the API response.
available_columns = [column for column in expected_columns if column in cdc_clean.columns]

# Subset the DataFrame to keep the fields needed for this project.
cdc_clean = cdc_clean[available_columns].copy()

# Rename columns into more readable project-specific names when those columns exist.
cdc_clean = cdc_clean.rename(columns={
    "stateabbr": "state_abbr",
    "statedesc": "state_name",
    "locationname": "county_name",
    "locationid": "county_fips",
    "data_value": "estimate_percent",
    "data_value_unit": "estimate_unit",
    "data_value_type": "estimate_type",
    "low_confidence_limit": "low_confidence_limit_percent",
    "high_confidence_limit": "high_confidence_limit_percent",
    "totalpopulation": "total_population"
})

# Print the selected and renamed columns to document Transformation Step #2.
print("Transformation Step #2 - Columns kept and renamed:")
print(cdc_clean.columns.tolist())

# Display the first few rows after selecting relevant fields.
display(cdc_clean.head())


Transformation Step #2 - Columns kept and renamed:
['year', 'state_abbr', 'state_name', 'county_name', 'county_fips', 'category', 'measure', 'measureid', 'estimate_percent', 'estimate_unit', 'estimate_type', 'low_confidence_limit_percent', 'high_confidence_limit_percent', 'total_population']


,year,state_abbr,state_name,county_name,county_fips,category,measure,measureid,estimate_percent,estimate_unit,estimate_type,low_confidence_limit_percent,high_confidence_limit_percent,total_population
0,2023,AL,Alabama,Autauga,01001,Health Outcomes,High blood pressure among adults,BPHIGH,38.2,%,Age-adjusted prevalence,33.9,42.4,60342
1,2023,AL,Alabama,Autauga,01001,Health Outcomes,High blood pressure among adults,BPHIGH,42.0,%,Crude prevalence,37.7,46.2,60342
2,2023,AL,Alabama,Autauga,01001,Health Outcomes,Coronary heart disease among adults,CHD,5.6,%,Age-adjusted prevalence,4.9,6.4,60342
3,2023,AL,Alabama,Autauga,01001,Health Outcomes,Coronary heart disease among adults,CHD,6.7,%,Crude prevalence,5.9,7.6,60342
4,2023,AL,Alabama,Autauga,01001,Health Outcomes,Diagnosed diabetes among adults,DIABETES,11.4,%,Age-adjusted prevalence,9.8,13.2,60342


### Transformation Step #3: Convert numeric fields

This step converts estimate values, confidence limits, and population fields from text into numeric data types. Numeric conversion is important because percentages and confidence limits cannot be properly summarized or compared when they remain stored as strings.


In [7]:
# Convert estimate values from text to numeric values so they can be analyzed correctly.
cdc_clean["estimate_percent"] = pd.to_numeric(cdc_clean["estimate_percent"], errors="coerce")

# Convert the low confidence limit from text to numeric values when the column exists.
if "low_confidence_limit_percent" in cdc_clean.columns:
    # Coerce invalid values to NaN so bad values do not break the notebook.
    cdc_clean["low_confidence_limit_percent"] = pd.to_numeric(cdc_clean["low_confidence_limit_percent"], errors="coerce")

# Convert the high confidence limit from text to numeric values when the column exists.
if "high_confidence_limit_percent" in cdc_clean.columns:
    # Coerce invalid values to NaN so bad values do not break the notebook.
    cdc_clean["high_confidence_limit_percent"] = pd.to_numeric(cdc_clean["high_confidence_limit_percent"], errors="coerce")

# Convert the total population field from text to numeric values when the column exists.
if "total_population" in cdc_clean.columns:
    # Coerce invalid values to NaN so population can be summarized accurately.
    cdc_clean["total_population"] = pd.to_numeric(cdc_clean["total_population"], errors="coerce")

# Print the updated data types to document Transformation Step #3.
print("Transformation Step #3 - Data types after numeric conversion:")
print(cdc_clean.dtypes)


Transformation Step #3 - Data types after numeric conversion:
year                                 str
state_abbr                           str
state_name                           str
county_name                          str
county_fips                          str
category                             str
measure                              str
measureid                            str
estimate_percent                 float64
estimate_unit                        str
estimate_type                        str
low_confidence_limit_percent     float64
high_confidence_limit_percent    float64
total_population                   int64
dtype: object


### Transformation Step #4: Select one estimate type for analysis

This step keeps only age-adjusted prevalence estimates. The CDC API includes both crude prevalence and age-adjusted prevalence for the same county and measure. Keeping one estimate type prevents the notebook from averaging two different statistical estimates together during the pivot step. Age-adjusted prevalence is used because it supports fairer comparisons across counties with different age distributions.

In [8]:
# Review the available estimate types before selecting one for analysis.
estimate_type_counts = cdc_clean["estimate_type"].value_counts()

# Print the estimate type counts so the choice is documented.
print("Available estimate types before filtering:")
print(estimate_type_counts)

# Keep only age-adjusted prevalence estimates for county comparison.
cdc_clean = cdc_clean[cdc_clean["estimate_type"] == "Age-adjusted prevalence"].copy()

# Print the shape after filtering to one estimate type.
print("Shape after keeping only age-adjusted prevalence estimates:", cdc_clean.shape)

Available estimate types before filtering:
estimate_type
Age-adjusted prevalence    14790
Crude prevalence           14790
Name: count, dtype: int64
Shape after keeping only age-adjusted prevalence estimates: (14790, 14)


### Transformation Step #5: Fix casing and inconsistent text values

This step standardizes state names, county names, state abbreviations, measure IDs, and measure descriptions. Consistent text formatting is especially important when data may later be merged with other sources by state, county, or FIPS code.


In [9]:
# Standardize state names by trimming extra spaces and applying title case.
cdc_clean["state_name"] = cdc_clean["state_name"].astype(str).str.strip().str.title()

# Standardize county names by trimming extra spaces and applying title case.
cdc_clean["county_name"] = cdc_clean["county_name"].astype(str).str.strip().str.title()

# Standardize state abbreviations by trimming spaces and applying uppercase formatting.
cdc_clean["state_abbr"] = cdc_clean["state_abbr"].astype(str).str.strip().str.upper()

# Standardize measure IDs by trimming spaces and applying uppercase formatting.
cdc_clean["measureid"] = cdc_clean["measureid"].astype(str).str.strip().str.upper()

# Standardize measure descriptions by trimming extra spaces.
cdc_clean["measure"] = cdc_clean["measure"].astype(str).str.strip()

# Print a sample of standardized text fields to document Transformation Step #5.
print("Transformation Step #5 - Standardized text fields preview:")
display(cdc_clean[["state_abbr", "state_name", "county_name", "measureid", "measure"]].head())


Transformation Step #5 - Standardized text fields preview:


,state_abbr,state_name,county_name,measureid,measure
0,AL,Alabama,Autauga,BPHIGH,High blood pressure among adults
2,AL,Alabama,Autauga,CHD,Coronary heart disease among adults
4,AL,Alabama,Autauga,DIABETES,Diagnosed diabetes among adults
6,AL,Alabama,Autauga,OBESITY,Obesity among adults
8,AL,Alabama,Autauga,STROKE,Stroke among adults


### Transformation Step #6: Identify and remove duplicates

This step checks for fully duplicated records and removes them. Duplicate records could overrepresent certain counties or measures, so reviewing duplicates protects the integrity of the cleaned dataset.


In [10]:
# Count duplicate rows before removing duplicates.
duplicate_count_before = cdc_clean.duplicated().sum()

# Remove fully duplicated rows from the CDC API data.
cdc_clean = cdc_clean.drop_duplicates().copy()

# Count duplicate rows after removing duplicates.
duplicate_count_after = cdc_clean.duplicated().sum()

# Print the duplicate counts to document Transformation Step #6.
print("Transformation Step #6 - Duplicate review:")
print("Duplicate rows before removal:", duplicate_count_before)
print("Duplicate rows after removal:", duplicate_count_after)
print("Shape after duplicate removal:", cdc_clean.shape)


Transformation Step #6 - Duplicate review:
Duplicate rows before removal: 0
Duplicate rows after removal: 0
Shape after duplicate removal: (14790, 14)


### Transformation Step #7: Review and filter missing critical values

This step removes rows that are missing required geographic identifiers, measure IDs, or prevalence estimates. These fields are necessary for creating a county-level dataset and for later merging with other project sources.


In [11]:
# Define the key fields that must exist for the cleaned county-level dataset.
required_fields = ["state_abbr", "state_name", "county_name", "county_fips", "measureid", "estimate_percent"]

# Count missing values in the required fields before filtering.
missing_before = cdc_clean[required_fields].isna().sum()

# Remove rows missing critical geography, measure, or estimate fields.
cdc_clean = cdc_clean.dropna(subset=required_fields).copy()

# Count missing values in the required fields after filtering.
missing_after = cdc_clean[required_fields].isna().sum()

# Print the missing-value review to document Transformation Step #7.
print("Transformation Step #7 - Missing value review for required fields before filtering:")
print(missing_before)
print("Transformation Step #7 - Missing value review for required fields after filtering:")
print(missing_after)
print("Shape after missing-value filtering:", cdc_clean.shape)


Transformation Step #7 - Missing value review for required fields before filtering:
state_abbr          0
state_name          0
county_name         5
county_fips         0
measureid           0
estimate_percent    5
dtype: int64
Transformation Step #7 - Missing value review for required fields after filtering:
state_abbr          0
state_name          0
county_name         0
county_fips         0
measureid           0
estimate_percent    0
dtype: int64
Shape after missing-value filtering: (14780, 14)


### Transformation Step #8: Identify outliers and bad data

This step checks whether any percentage estimates fall outside the logical 0 to 100 range. Removing impossible percentage values helps ensure the final dataset remains valid for interpretation.


In [12]:
# Identify records with estimate percentages outside the expected 0 to 100 range.
outlier_mask = (cdc_clean["estimate_percent"] < 0) | (cdc_clean["estimate_percent"] > 100)

# Count the number of out-of-range estimate values before filtering.
outlier_count = outlier_mask.sum()

# Keep only records with estimate percentages between 0 and 100.
cdc_clean = cdc_clean.loc[~outlier_mask].copy()

# Print the outlier review to document Transformation Step #8.
print("Transformation Step #8 - Out-of-range estimate review:")
print("Number of estimate values outside 0 to 100:", outlier_count)
print("Shape after outlier filtering:", cdc_clean.shape)


Transformation Step #8 - Out-of-range estimate review:
Number of estimate values outside 0 to 100: 0
Shape after outlier filtering: (14780, 14)


### Transformation Step #9: Format data into a more readable wide format

This step reshapes the API data from long format into a county-level wide format. Instead of having one row per county-measure combination, the final dataset has one row per county with separate columns for diabetes, obesity, high blood pressure, coronary heart disease, and stroke.


In [13]:
# Create a smaller DataFrame for pivoting the chronic disease measures into separate columns.
pivot_input = cdc_clean[[
    "year",
    "state_abbr",
    "state_name",
    "county_name",
    "county_fips",
    "measureid",
    "estimate_percent"
]].copy()

# Pivot the data so each county has one row and each selected health measure has its own column.
cdc_wide = pivot_input.pivot_table(
    index=["year", "state_abbr", "state_name", "county_name", "county_fips"],
    columns="measureid",
    values="estimate_percent",
    aggfunc="mean"
).reset_index()

# Remove the columns index name created during the pivot operation.
cdc_wide.columns.name = None

# Rename measure columns into more readable final dataset names.
cdc_wide = cdc_wide.rename(columns={
    "DIABETES": "diabetes_percent",
    "OBESITY": "obesity_percent",
    "BPHIGH": "high_blood_pressure_percent",
    "CHD": "coronary_heart_disease_percent",
    "STROKE": "stroke_percent"
})

# Print the shape of the reshaped county-level dataset to document Transformation Step #9.
print("Transformation Step #9 - Shape after reshaping from long to wide format:", cdc_wide.shape)

# Display the first few rows after reshaping.
display(cdc_wide.head())


Transformation Step #9 - Shape after reshaping from long to wide format: (2956, 10)


,year,state_abbr,state_name,county_name,county_fips,high_blood_pressure_percent,coronary_heart_disease_percent,diabetes_percent,obesity_percent,stroke_percent
0,2023,AK,Alaska,Aleutians East,02013,33.8,6.4,12.9,29.8,3.6
1,2023,AK,Alaska,Aleutians West,02016,32.2,5.7,11.4,30.5,3.1
2,2023,AK,Alaska,Anchorage,02020,31.4,5.1,8.8,35.0,2.8
3,2023,AK,Alaska,Bethel,02050,39.7,8.6,14.7,39.4,5.4
4,2023,AK,Alaska,Bristol Bay,02060,31.2,5.9,9.1,34.5,3.2


### Transformation Step #10: Create readable summary fields

This step creates a simple average chronic disease burden percentage and a readable burden category. The category is not a clinical diagnosis. It is a descriptive data preparation feature that helps summarize county-level health context for later exploratory analysis.


In [14]:
# Create a list of percentage columns for summary feature creation.
health_percent_columns = [
    "diabetes_percent",
    "obesity_percent",
    "high_blood_pressure_percent",
    "coronary_heart_disease_percent",
    "stroke_percent"
]

# Keep only percentage columns that exist in the reshaped dataset.
available_health_columns = [column for column in health_percent_columns if column in cdc_wide.columns]

# Calculate an average chronic disease burden score across the selected measures.
cdc_wide["average_chronic_disease_burden_percent"] = cdc_wide[available_health_columns].mean(axis=1)

# Create a readable risk category based on the county's average chronic disease burden.
cdc_wide["chronic_disease_burden_category"] = pd.cut(
    cdc_wide["average_chronic_disease_burden_percent"],
    bins=[-np.inf, 20, 30, np.inf],
    labels=["Lower", "Moderate", "Higher"]
)

# Print the created feature names to document Transformation Step #10.
print("Transformation Step #10 - Added summary burden fields:")
print(["average_chronic_disease_burden_percent", "chronic_disease_burden_category"])

# Display the first few rows with the new summary fields.
display(cdc_wide[["state_abbr", "county_name", "average_chronic_disease_burden_percent", "chronic_disease_burden_category"]].head())


Transformation Step #10 - Added summary burden fields:
['average_chronic_disease_burden_percent', 'chronic_disease_burden_category']


,state_abbr,county_name,average_chronic_disease_burden_percent,chronic_disease_burden_category
0,AK,Aleutians East,17.30,Lower
1,AK,Aleutians West,16.58,Lower
2,AK,Anchorage,16.62,Lower
3,AK,Bethel,21.56,Moderate
4,AK,Bristol Bay,16.78,Lower


### Final human-readable API dataset

The final dataset is sorted, rounded, and formatted so it can be reviewed by a person without needing to interpret the original API structure. This satisfies the milestone requirement to print a human-readable dataset after all transformations are complete.


In [15]:
# Sort the final dataset by state and county so it is easier to read.
final_api_dataset = cdc_wide.sort_values(by=["state_name", "county_name"]).reset_index(drop=True)

# Round percentage columns to one decimal place for cleaner presentation.
for column in available_health_columns + ["average_chronic_disease_burden_percent"]:
    # Round each percentage column if it exists in the final dataset.
    if column in final_api_dataset.columns:
        # Apply rounding to improve readability in the final printed output.
        final_api_dataset[column] = final_api_dataset[column].round(1)

# Print the final dataset shape after all transformations.
print("Final human-readable API dataset shape:", final_api_dataset.shape)

# Display the first 20 rows of the final human-readable API dataset.
display(final_api_dataset.head(20))


Final human-readable API dataset shape: (2956, 12)


,year,state_abbr,state_name,county_name,county_fips,high_blood_pressure_percent,coronary_heart_disease_percent,diabetes_percent,obesity_percent,stroke_percent,average_chronic_disease_burden_percent,chronic_disease_burden_category
0,2023,AL,Alabama,Autauga,01001,38.2,5.6,11.4,39.5,3.2,19.6,Lower
1,2023,AL,Alabama,Baldwin,01003,37.0,5.5,10.4,35.0,2.9,18.2,Lower
2,2023,AL,Alabama,Barbour,01005,45.4,7.4,16.5,47.1,4.7,24.2,Moderate
3,2023,AL,Alabama,Bibb,01007,42.4,7.0,13.9,43.7,4.0,22.2,Moderate
4,2023,AL,Alabama,Blount,01009,36.0,6.3,11.3,36.0,3.3,18.6,Lower
5,2023,AL,Alabama,Bullock,01011,52.4,8.8,21.3,51.0,6.2,27.9,Moderate
6,2023,AL,Alabama,Butler,01013,44.8,6.7,15.2,40.7,4.3,22.3,Moderate
7,2023,AL,Alabama,Calhoun,01015,41.5,6.4,12.9,39.1,3.6,20.7,Moderate
8,2023,AL,Alabama,Chambers,01017,42.8,6.5,14.5,41.6,4.1,21.9,Moderate
9,2023,AL,Alabama,Cherokee,01019,37.5,6.6,11.8,39.3,3.5,19.7,Lower


### Geographic level check: county level vs. state level

The main cleaned dataset is county-level because each row represents a county identified by county name and FIPS code. This additional state summary shows that the county-level API data can be aggregated to the state level if needed. This is important because prior milestones involved multiple sources that may not always exist at the same geographic level.


In [16]:
# Create a state-level summary to check that the county-level data can also be interpreted at a higher level.
state_summary = final_api_dataset.groupby(["state_abbr", "state_name"], as_index=False).agg(
    # Count the number of counties represented for each state.
    county_count=("county_fips", "nunique"),

    # Calculate the average chronic disease burden for each state.
    avg_chronic_disease_burden=("average_chronic_disease_burden_percent", "mean"),

    # Calculate the average diabetes percentage for each state.
    avg_diabetes_percent=("diabetes_percent", "mean"),

    # Calculate the average obesity percentage for each state.
    avg_obesity_percent=("obesity_percent", "mean")
)

# Round the state-level summary values for readability.
state_summary["avg_chronic_disease_burden"] = state_summary["avg_chronic_disease_burden"].round(1)
state_summary["avg_diabetes_percent"] = state_summary["avg_diabetes_percent"].round(1)
state_summary["avg_obesity_percent"] = state_summary["avg_obesity_percent"].round(1)

# Print a message explaining the level check.
print("Geographic level check - county-level data summarized to state level:")

# Display the first 15 state-level summary rows.
display(state_summary.head(15))


Geographic level check - county-level data summarized to state level:


,state_abbr,state_name,county_count,avg_chronic_disease_burden,avg_diabetes_percent,avg_obesity_percent
0,AK,Alaska,30,17.7,10.4,35.3
1,AL,Alabama,67,21.6,13.9,41.9
2,AR,Arkansas,75,20.8,12.4,41.1
3,AZ,Arizona,15,16.9,11.0,33.7
4,CA,California,58,15.6,10.3,30.0
5,CO,Colorado,64,14.1,8.6,27.1
6,CT,Connecticut,9,15.0,8.4,30.6
7,DC,District Of Columbia,1,14.2,8.5,25.1
8,DE,Delaware,3,18.0,11.0,37.2
9,FL,Florida,67,17.8,11.4,34.8


## Ethical implications of data wrangling

The data wrangling process changed the CDC PLACES API data by standardizing headers, selecting relevant fields, converting numeric values, fixing text casing, removing duplicates, filtering missing critical fields, checking for impossible percentage values, reshaping the data from long to wide format, and creating a descriptive chronic disease burden summary. The data is public, aggregated, and county-level, so it does not contain individual patient records. However, healthcare data still requires careful interpretation because county-level health estimates can influence how communities are described. Legal and regulatory concerns include avoiding any implication that this public aggregate dataset contains protected health information under HIPAA. The main risk created by the transformations is that the average burden score could oversimplify complex health conditions or make counties appear inefficient or unhealthy without considering access to care, poverty, age distribution, insurance coverage, or other social factors. The notebook assumes that CDC PLACES estimates are credible for county-level comparison and that selected measures are relevant to healthcare utilization context. The data was sourced ethically from a public CDC API. To mitigate ethical concerns, the notebook documents each transformation, avoids causal claims, labels the burden category as descriptive rather than diagnostic, and keeps the final interpretation focused on preparation for analysis rather than judgment of communities.


## Conclusion

This Milestone 4 notebook demonstrates how API data can be pulled, validated, cleaned, reshaped, and prepared for later analysis. The revised version focuses on the CDC PLACES API because that was the API source identified in the project plan and because the previous CMS API connection produced repeated HTTP errors. The notebook performs more than the required five transformations, including header standardization, field selection, numeric conversion, text cleaning, duplicate review, missing-value filtering, outlier checking, reshaping, and feature creation.

The key data preparation lesson is that cleaning data is not only about making the dataset look neat. It also involves validating the source, checking whether fields exist, confirming the geographic level of the data, documenting assumptions, and avoiding misleading interpretations. For this project, county-level chronic disease estimates can provide useful context for later healthcare utilization analysis, but they should not be used alone to claim that a county is inefficient or that one condition directly causes higher utilization. The cleaned dataset is now in a human-readable structure that supports responsible exploratory analysis in later project work.


## References

Centers for Disease Control and Prevention. (2026). *PLACES: Local data for better health*. U.S. Department of Health and Human Services. https://www.cdc.gov/places/

Centers for Disease Control and Prevention. (2024). *PLACES data portal*. U.S. Department of Health and Human Services. https://www.cdc.gov/places/tools/data-portal.html

Centers for Disease Control and Prevention. (n.d.). *PLACES: Local data for better health, county data 2025 release* [Data set]. Data.CDC.gov. https://data.cdc.gov/500-Cities-Places/PLACES-Local-Data-for-Better-Health-County-Data-20/swc5-untb

Centers for Disease Control and Prevention. (2025). *Methodology: PLACES*. U.S. Department of Health and Human Services. https://www.cdc.gov/places/methodology/

Tyler Technologies. (n.d.). *Socrata Open Data API documentation*. https://dev.socrata.com/